# Volet B — Recherche de clauses (m2) et LSTM (m3)

**Ce que fait ce notebook** (il n'appelle que `src/ledgar/`, sans logique propre)
1. **Recherche** (`ledgar.retrieval`, jalon m2) : banque = les 60 000 clauses du train, encodées par `all-MiniLM-L6-v2`, vecteurs normalisés, top-k par cosinus. Comparaison avec TF-IDF, recherche hybride (sémantique + classe prédite + durée trouvée par regex), vérification à la main.
2. **LSTM** (`ledgar.lstm`, jalon m3) : BiLSTM + GloVe 300d, boucle d'entraînement PyTorch écrite à la main, early stopping sur la macro-F1 de validation, analyse d'erreurs.

**Règles** : tous les choix sur la validation, le test une seule fois avec l'époque retenue. Doublons exacts train/test (2,8 %) retirés du classement de la recherche et rapportés à part pour le LSTM.

Exécution > Modifier le type d'exécution > **GPU T4**.

In [ ]:
import os
# Le repo est privé : ajouter un token GitHub (droit "repo") dans les secrets Colab (icône clé à gauche), nom GH_TOKEN
from google.colab import userdata
TOKEN = userdata.get("GH_TOKEN")
if not os.path.exists("ledgar-contract-review"):
    !git clone -b feat/retrieval https://{TOKEN}@github.com/saouss-07/ledgar-contract-review.git
%cd ledgar-contract-review
!pip install -q -r requirements.txt && pip install -q -e .
import torch; print("GPU :", torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 1 — Recherche : évaluation (m2)

Une clause retrouvée est **pertinente** si elle a le même type que la clause requête. Requêtes = validation et test, banque = train.

In [ ]:
!python -m ledgar.retrieval

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
res = pd.read_csv("results/tables/retrieval_results.csv")
res[res.twins_removed].round(3)

**À lire** : `minilm + classe predite` ne garde que les clauses de la classe prédite par la baseline ; sa P@k est donc bornée par l'accuracy du classifieur. Ce n'est pas une mesure de la qualité des vecteurs seuls, c'est la mesure du système tel que l'avocat l'utiliserait.

In [ ]:
per_class = pd.read_csv("results/tables/retrieval_p5_per_class.csv")
print("types de clauses les plus mal servis par la recherche sémantique seule :")
per_class.head(10)

## 2 — Recherche : vérification à la main

3 clauses collées (tirées du test) et 5 questions d'avocat, en sémantique seule puis en hybride. Colonne `label_name` = type de la clause retrouvée.

In [ ]:
ex = pd.read_csv("results/tables/retrieval_examples.csv")
for (kind, q), g in ex.groupby(["query_kind", "query"], sort=False):
    print(f"
## {kind} | {q[:100]}")
    print(f"   vrai type : {g.query_label.iloc[0]} | filtres : {g.filter_label.iloc[0]}, {g.filter_duration.iloc[0]}")
    for _, r in g.iterrows():
        print(f"   {r['rank']}. {r.score:.3f} [{r.label_name}] {r.text[:110]}")

### Essayer sa propre question

In [ ]:
import numpy as np
from ledgar.data import load_ledgar
from ledgar.retrieval import ClauseBank, baseline_classifier, cached_embeddings, load_encoder

ds, names = load_ledgar()
encoder = load_encoder()
bank = ClauseBank(ds["train"]["text"], ds["train"]["label"], names,
                  cached_embeddings(encoder, ds["train"]["text"], "minilm_train"), encoder,
                  baseline_classifier(ds["train"]))
bank.search("show me termination clauses with a 30-day notice period", k=5, label="auto", duration="auto")

## 3 — LSTM : entraînement (m3)

Boucle écrite à la main dans `ledgar.lstm.train_one_epoch` : forward, cross-entropy, backward, clipping, pas d'AdamW.

In [ ]:
!python -m ledgar.lstm

In [ ]:
hist = pd.read_csv("results/tables/lstm_history.csv")
ax = hist.plot(x="epoch", y=["val_macro_f1", "val_accuracy"], marker="o", title="LSTM : validation par époque")
hist.round(4)

In [ ]:
lstm = pd.read_csv("results/tables/lstm_results.csv")
base = pd.read_csv("results/tables/baseline_results.csv")
pd.concat([base[["model", "split", "accuracy", "macro_f1"]], lstm[["model", "split", "accuracy", "macro_f1"]]]).round(4)

## 4 — LSTM : analyse d'erreurs

F1 par classe face à la baseline, et paires de classes les plus confondues.

In [ ]:
f1 = pd.read_csv("results/tables/lstm_f1_per_class.csv")[["label_name", "support", "f1"]].merge(
    pd.read_csv("results/tables/baseline_f1_per_class.csv")[["label_name", "f1"]], on="label_name", suffixes=("_lstm", "_baseline"))
f1["ecart"] = f1.f1_lstm - f1.f1_baseline
print("10 classes les plus faibles du LSTM :")
display(f1.sort_values("f1_lstm").head(10).round(3))
print("F1 moyenne par tranche de support (rares = faible support) :")
f1["tranche"] = pd.qcut(f1.support, 3, labels=["rare", "moyenne", "fréquente"])
f1.groupby("tranche", observed=True)[["f1_lstm", "f1_baseline"]].mean().round(3)

In [ ]:
pd.read_csv("results/tables/lstm_top_confusions.csv")

In [ ]:
from ledgar.metrics import load_predictions, contingency, mcnemar, paired_bootstrap
a, b = load_predictions("lstm", "test"), load_predictions("baseline", "test")
assert (a.idx.values == b.idx.values).all()
_, only_lstm, only_base, _ = contingency(a.label, a.pred, b.pred)
print("McNemar (chi2, p) :", mcnemar(only_lstm, only_base))
print("bootstrap macro-F1 LSTM - baseline (obs, IC95 bas, haut) :", paired_bootstrap(a.label, a.pred, b.pred, 100))

## 5 — Sauvegarder les résultats

Les tableaux et prédictions sont dans `results/`. Les télécharger (ou les pousser sur la branche) : ils sont petits et suivent le format commun `idx, label, pred, confidence`.

In [ ]:
!zip -qr results_B.zip results/tables/retrieval_* results/tables/lstm_* results/predictions/lstm_*
from google.colab import files; files.download("results_B.zip")